# Imports

In [21]:
import pandas as pd
import numpy as np
import json

# Extract all data And Aggregate Hourly 

In [22]:
url_file_path = "../../data/URLs.json"

with open(url_file_path, 'r') as file:
    url_list = json.load(file)["entries"]


#### Stationwise Amenity Count Data

In [23]:
stationwise_amenity_counts = pd.read_csv("../../data/stationwise_amenity_count.csv")


#### Stationwise Nearest Amenity Distance Data

In [24]:
stationwise_nearest_amenity_dist = pd.read_csv("../../data/stationwise_amenity_dist.csv")

#### Station location data

In [25]:
station_id_lat_lon = pd.read_csv("../../data/id_lat_lon.csv")

#### UK holiday data

In [26]:
uk_holidays = pd.read_csv("../../data/uk_holiday.csv")
uk_holidays = uk_holidays.drop(columns=["notes", "bunting", "title"], axis=1)
uk_holidays["date"] = pd.to_datetime(uk_holidays["date"])

In [29]:
month_map = {
    "Jan":1,
    "Feb":2,
    "Mar":3,
    "Apr":4,
    "May":5,
    "Jun":6,
    "Jul":7,
    "Aug":8,
    "Sep":9,
    "Oct":10,
    "Nov":11,
    "Dec":12
}

for m in month_map.keys():
    month = m # Ex: Jan, Feb, Mar

    print(f"Processing {month} data...")

    urls_for_month = []

    for url in url_list:
        if month in  url["url"]:
            urls_for_month.append(url["url"])

            
    data = pd.concat([pd.read_csv(file) for file in urls_for_month], ignore_index=True)

    drop_column_list = ["Number", "End date", "End station number", "End station", "Bike number", "Bike model", "Total duration", "Start station"]

    # Remove Records with duration less than 1 minute
    data = data[data["Total duration (ms)"] > 60000]
    drop_column_list.append("Total duration (ms)")

    data["Start date"] = pd.to_datetime(data["Start date"])
    Month_data = data[data["Start date"].dt.month == month_map[month]]
    Month_data = Month_data.drop(columns=drop_column_list, axis=1)


    stations_with_coords = station_id_lat_lon[
        station_id_lat_lon['latitude'].notna() & 
        station_id_lat_lon['longitude'].notna()
    ]

    Month_data = pd.merge(
        Month_data,
        station_id_lat_lon,
        left_on="Start station number",
        right_on="start_station_id",
        how="inner"
    )
    Month_data = Month_data.drop(columns="start_station_id", axis=1)


    Month_data["s_hour"] = Month_data["Start date"].dt.hour
    Month_data["s_date"] = pd.to_datetime(Month_data["Start date"].dt.date)
    Month_data["day_of_week"] = Month_data["s_date"].dt.day_of_week
    Month_data["week_of_month"] = (Month_data["s_date"].dt.day - 1) // 7 + 1   
    Month_data["is_weekend"] = Month_data["day_of_week"].isin([5,6])
    Month_data["weekend_or_weekday_sdate"] = Month_data["is_weekend"].map({True: "weekend", False: "weekday"})
    Month_data = Month_data.drop(columns=["Start date", "is_weekend"], axis=1)

    all_dates = Month_data["s_date"].unique()
    all_hours = Month_data["s_hour"].unique()
    all_stations = Month_data["Start station number"].unique()

    full_index = pd.MultiIndex.from_product(
        [all_dates, all_hours, all_stations],
        names=["s_date", "s_hour", "Start station number"]
    )

    full_grid = pd.DataFrame(index=full_index).reset_index()

    aggregated = Month_data.groupby(["s_date", "s_hour", "Start station number"]).agg({
        'day_of_week': 'first',
        'week_of_month': 'first',
        'weekend_or_weekday_sdate': 'first'
    }).reset_index()

    aggregated["count"] = Month_data.groupby(["s_date", "s_hour", "Start station number"]).size().reset_index(name='count')['count']

    Month_data = full_grid.merge(aggregated, on=["s_date", "s_hour", "Start station number"], how='left')

    Month_data["day_of_week"] = Month_data["s_date"].dt.day_of_week
    Month_data["week_of_month"] = (Month_data["s_date"].dt.day - 1) // 7 + 1   
    Month_data["is_weekend"] = Month_data["day_of_week"].isin([5,6])
    Month_data["weekend_or_weekday_sdate"] = Month_data["is_weekend"].map({True: "weekend", False: "weekday"})
    Month_data = Month_data.drop(columns="is_weekend", axis=1)
    Month_data["count"] = Month_data["count"].fillna(0)


    holiday_indicator = uk_holidays[['date']].copy()
    holiday_indicator['is_holiday'] = True

    Month_data = Month_data.merge(holiday_indicator, 
                left_on='s_date', 
                right_on='date', 
                how='left')

    Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)

    Month_data = Month_data.drop('date', axis=1)


    time_of_day_conditions = [
        (Month_data["s_hour"] >= 0) & (Month_data["s_hour"] <= 5),
        (Month_data["s_hour"] >= 6) & (Month_data["s_hour"] <= 11),
        (Month_data["s_hour"] >= 12) & (Month_data["s_hour"] <= 17),
        (Month_data["s_hour"] >= 18) & (Month_data["s_hour"] <= 21),
        (Month_data["s_hour"] >= 22) & (Month_data["s_hour"] <= 23),
        
    ]

    time_of_day_choices = ["night", "morning", "day", "evening", "night"]

    Month_data["time_of_day"] = np.select(condlist=time_of_day_conditions, choicelist=time_of_day_choices, default="Unknown")

    peak_off_peak_conditions = [
        (Month_data["weekend_or_weekday_sdate"] == "weekend") & (Month_data["s_hour"].between(10,17)),
        (Month_data["weekend_or_weekday_sdate"] == "weekday") & (Month_data["is_holiday"] == True) & (Month_data["s_hour"].between(10,17)),
        (Month_data["weekend_or_weekday_sdate"] == "weekday") & (Month_data["s_hour"].between(7,10)) & (Month_data["s_hour"].between(17,19))
    ]

    peak_off_peak_choices = [1, 1, 1]

    Month_data["peak_off_peak"] = np.select(condlist=peak_off_peak_conditions, choicelist=peak_off_peak_choices, default=0)

    Month_data = Month_data.merge(
        stationwise_amenity_counts,
        left_on="Start station number",
        right_on="start_station_number",
        how="left"
    )

    amenity_distance_col_list = [
            "start_station_number", 
            'dist_to_nearest_cafe',
            'dist_to_nearest_atm',
            'dist_to_nearest_pub',
            'dist_to_nearest_school',
            'dist_to_nearest_university',
            'dist_to_nearest_college',
            'dist_to_nearest_bank', 
            'dist_to_nearest_post_office',
            'dist_to_nearest_library',
            'dist_to_nearest_cinema',
            'dist_to_nearest_supermarket',
            'dist_to_nearest_station',
            'dist_to_nearest_platform',
            'dist_to_nearest_stop_position',
            'dist_to_nearest_railway_station',
            'dist_to_nearest_bus_stop'
            ]

    Month_data = Month_data.merge(
        stationwise_nearest_amenity_dist[[col for col in amenity_distance_col_list]],
        left_on="Start station number",
        right_on="start_station_number",
        how="left"
    )
    Month_data = Month_data.rename(columns={"start_station_number_x":"start_station_number"})

    Month_data = Month_data.drop(columns=["Start station number","start_station_number_y"], axis=1)


    output_filename_map = {
        "Jan":"January_h",
        "Feb":"February_h",
        "Mar":"March_h",
        "Apr":"April_h",
        "May":"May_h",
        "Jun":"June_h",
        "Jul":"July_h",
        "Aug":"August_h",
        "Sep":"September_h",
        "Oct":"October_h",
        "Nov":"November_h",
        "Dec":"December_h"
    }

    Month_data.to_csv(f"../../data/dataset_hourly/{output_filename_map[month]}.csv", index=False)


Processing Jan data...


/var/folders/69/5y36t1_d6cq93tyzkyb102x80000gn/T/ipykernel_8013/1522617139.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)


Processing Feb data...


/var/folders/69/5y36t1_d6cq93tyzkyb102x80000gn/T/ipykernel_8013/1522617139.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)


Processing Mar data...


...

/var/folders/69/5y36t1_d6cq93tyzkyb102x80000gn/T/ipykernel_8013/1522617139.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)


Processing Nov data...


/var/folders/69/5y36t1_d6cq93tyzkyb102x80000gn/T/ipykernel_8013/1522617139.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)


Processing Dec data...


/var/folders/69/5y36t1_d6cq93tyzkyb102x80000gn/T/ipykernel_8013/1522617139.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Month_data['is_holiday'] = Month_data['is_holiday'].fillna(False)


In [30]:
Month_data.columns

Index(['s_date', 's_hour', 'day_of_week', 'week_of_month',
       'weekend_or_weekday_sdate', 'count', 'is_holiday', 'time_of_day',
       'peak_off_peak', 'start_station_number', 'longitude', 'latitude',
       'cafe_count_5min_walk', 'atm_count_5min_walk', 'pub_count_5min_walk',
       'school_count_5min_walk', 'university_count_5min_walk',
       'college_count_5min_walk', 'bank_count_5min_walk',
       'post_office_count_5min_walk', 'library_count_5min_walk',
       'cinema_count_5min_walk', 'supermarket_count_5min_walk',
       'station_count_5min_walk', 'platform_count_5min_walk',
       'stop_position_count_5min_walk', 'railway_station_count_5min_walk',
       'highway_bus_stop_count_5min_walk', 'dist_to_nearest_cafe',
       'dist_to_nearest_atm', 'dist_to_nearest_pub', 'dist_to_nearest_school',
       'dist_to_nearest_university', 'dist_to_nearest_college',
       'dist_to_nearest_bank', 'dist_to_nearest_post_office',
       'dist_to_nearest_library', 'dist_to_nearest_cinema',

# Aggregate Monthly

In [33]:
filename_map = {
    "Jan":"January",
    "Feb":"February",
    "Mar":"March",
    "Apr":"April",
    "May":"May",
    "Jun":"June",
    "Jul":"July",
    "Aug":"August",
    "Sep":"September",
    "Oct":"October",
    "Nov":"November",
    "Dec":"December"
}

for m in filename_map.keys():

    month = m

    print(f"Processing {month} data...")

    data = pd.read_csv(f"../../data/dataset_hourly/{filename_map[month]}_h.csv")

    columns_to_keep = [col for col in data.columns if col not in [
        'peak_off_peak', 'is_holiday', 'time_of_day', 'day_of_week',
        'week_of_month', 'weekend_or_weekday_sdate', 's_date', 's_hour'
    ]]

    agg_dict = {'count': 'sum'}
    for col in columns_to_keep:
        if col != 'count' and col != 'start_station_number':
            agg_dict[col] = 'first'
            
    aggregated_data = data[columns_to_keep].groupby('start_station_number').agg(agg_dict).reset_index()

    aggregated_data.to_csv(f"../../data/dataset_monthly/{filename_map[month]}_m.csv", index = False)

Processing Jan data...
Processing Feb data...
Processing Mar data...
Processing Apr data...
Processing May data...
Processing Jun data...
Processing Jul data...
Processing Aug data...
Processing Sep data...
Processing Oct data...
Processing Nov data...
Processing Dec data...
